# CatBoostRanker final pipeline


In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import sys
from pathlib import Path

import joblib
import numpy as np
import optuna
import pandas as pd
from catboost import CatBoostRanker, Pool
from implicit.als import AlternatingLeastSquares
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

CWD = Path.cwd()
ROOT_DIR = CWD if (CWD / "src").exists() else CWD.parent
sys.path.append(str(ROOT_DIR))

from src.temporal_split import TemporalSplit
from src.models.recommender import Recommender
from src.metrics.metrics import recall_at_k, ndcg_at_k

DATA_DIR = ROOT_DIR / "notebook" / "data" / "11092026"
if not DATA_DIR.exists():
    DATA_DIR = CWD / "data" / "11092026"


## Data and temporal splits


In [ ]:
df_users = pd.read_parquet(DATA_DIR / "users.parquet")
df_movies = pd.read_parquet(DATA_DIR / "movies.parquet")
df_interactions = pd.read_parquet(DATA_DIR / "interactions.parquet")

encoders = joblib.load(DATA_DIR / "encoders.pkl")
metadata = joblib.load(DATA_DIR / "metadata.pkl")
user_encoder = encoders["user_encoder"]
movie_encoder = encoders["movie_encoder"]
n_users = metadata["n_users"]

df_interactions = df_interactions.copy()
df_interactions["user_id"] = user_encoder.inverse_transform(df_interactions["user_idx"])
df_interactions["movie_id"] = movie_encoder.inverse_transform(df_interactions["movie_idx"])

df_users = df_users.copy()
df_movies = df_movies.copy()
df_users["gender"] = df_users["gender"].astype(str)
df_users["age_group"] = df_users["age_group"].astype(str)
df_users["occupation"] = df_users["occupation"].astype(str)
df_movies["year"] = pd.to_numeric(df_movies["title"].str.extract(r"\((\d{4})\)")[0], errors="coerce")


In [ ]:
splitter = TemporalSplit()
train, val, test = splitter.split_to_three(df_interactions)
rank_history, rank_target = splitter.split_to_two(train)

val_history = train.copy()
val_target = val.copy()
test_history = pd.concat([train, val], ignore_index=True)
test_target = test.copy()


## Retrieval


In [ ]:
def fit_als(
    history,
    n_users,
    factors=25,
    regularization=0.3,
    iterations=30
):
    movie_ids = history["movie_id"].unique()
    movie_to_als = {
        movie_id: idx
        for idx, movie_id in enumerate(movie_ids)
    }

    als_to_movie = {
        idx: movie_id
        for movie_id, idx in movie_to_als.items()
    }

    positive = history.loc[
        history["rating"] >= 4,
        ["user_idx", "movie_id", "rating"]
    ].copy()

    positive["als_movie_idx"] = positive["movie_id"].map(movie_to_als)
    values = (positive["rating"] - 3).values.astype(np.float32)

    user_item = csr_matrix(
        (
            values,
            (positive["user_idx"], positive["als_movie_idx"])
        ),
        shape=(n_users, len(movie_to_als)),
        dtype=np.float32
    )

    model = AlternatingLeastSquares(
        factors=factors,
        regularization=regularization,
        iterations=iterations,
        num_threads=6,
        random_state=42
    )

    model.fit(user_item, show_progress=True)

    return {
        "model": model,
        "user_item": user_item,
        "movie_to_als": movie_to_als,
        "als_to_movie": als_to_movie
    }

def fit_item_similarity(
    history,
    n_users,
    k_neighbors=100,
):
    movie_ids = history["movie_id"].unique()

    movie_to_sim = {
        movie_id: idx
        for idx, movie_id in enumerate(movie_ids)
    }

    sim_to_movie = {
        idx: movie_id
        for movie_id, idx in movie_to_sim.items()
    }

    positive = history.loc[
        history["rating"] >= 4,
        ["user_idx", "movie_id", "rating"]
    ].copy()

    positive["sim_movie_idx"] = positive["movie_id"].map(movie_to_sim)
    values = (positive["rating"] - 3).values.astype(np.float32)

    user_movie = csr_matrix(
        (
            values,
            (positive["user_idx"], positive["sim_movie_idx"])
        ),
        shape=(n_users, len(movie_to_sim)),
        dtype=np.float32
    )

    movie_user = normalize(user_movie.T.tocsr(), axis=1)

    model = NearestNeighbors(
        metric="cosine",
        algorithm="brute",
        n_jobs=-1,
    )

    model.fit(movie_user)

    n_neighbors = min(k_neighbors + 1, movie_user.shape[0])

    distances, indices = model.kneighbors(
        movie_user,
        n_neighbors=n_neighbors,
    )

    neighbors = {
        movie_idx: (
            indices[movie_idx][1:],
            1.0 - distances[movie_idx][1:]
        )
        for movie_idx in range(movie_user.shape[0])
    }

    return {
        "neighbors": neighbors,
        "movie_to_sim": movie_to_sim,
        "sim_to_movie": sim_to_movie,
    }


In [ ]:
def build_candidates(
    users,
    history,
    als_artifacts,
    sim_artifacts,
    popularity_model,
    k_als=500,
    k_sim=100,
    k_pop=100,
    max_sim_seeds=50
):
    seen = history.groupby("user_idx")["movie_id"].apply(set).to_dict()

    liked_recent = (
        history.loc[history["rating"] >= 4]
        .sort_values(["user_idx", "timestamp"])
        .groupby("user_idx")
        .tail(max_sim_seeds)
        .groupby("user_idx")["movie_id"]
        .apply(list)
        .to_dict()
    )

    als_model = als_artifacts["model"]
    user_item = als_artifacts["user_item"]
    als_to_movie = als_artifacts["als_to_movie"]

    neighbors = sim_artifacts["neighbors"]
    movie_to_sim = sim_artifacts["movie_to_sim"]
    sim_to_movie = sim_artifacts["sim_to_movie"]

    rows = []

    for user_idx in users:
        candidates = {}
        watched = seen.get(user_idx, set())

        request_k = min(k_als + len(watched), user_item.shape[1])
        ids, scores = als_model.recommend(
            userid=int(user_idx),
            user_items=user_item[int(user_idx)],
            N=request_k,
            filter_already_liked_items=True,
        )

        als_rank = 0

        for movie_idx, score in zip(ids, scores):

            movie_id = als_to_movie[int(movie_idx)]

            if movie_id in watched:
                continue

            als_rank += 1
            candidates.setdefault(movie_id, {})
            candidates[movie_id].update(als_score=float(score), als_rank=als_rank, from_als=1)

            if als_rank >= k_als:
                break

        sim_scores = {}

        for seed_movie_id in liked_recent.get(user_idx, []):
            seed_idx = movie_to_sim.get(seed_movie_id)
            if seed_idx is None:
                continue

            neighbor_ids, neighbor_scores = neighbors[seed_idx]

            for neighbor_idx, score in zip(neighbor_ids, neighbor_scores):
                movie_id = sim_to_movie[neighbor_idx]

                if movie_id in watched:
                    continue

                sim_scores[movie_id] = max(sim_scores.get(movie_id, 0), score)

        sim_top = sorted(sim_scores.items(), key=lambda x: x[1], reverse=True)[:k_sim]

        for rank, (movie_id, score) in enumerate(sim_top, start=1):
            candidates.setdefault(movie_id, {})
            candidates[movie_id].update(sim_score=float(score), sim_rank=rank, from_sim=1)

        pop_recs = popularity_model.recommend(
            user_idx, 
            exclude_movies=seen,
            k=k_pop
        )

        for rank, (movie_id, score) in enumerate(pop_recs, start=1):
            candidates.setdefault(movie_id, {})
            candidates[movie_id].update(pop_score=float(score), pop_rank=rank, from_pop=1)            

        for movie_id, features in candidates.items():

            rrf = 0.

            if features.get("from_als"):
                rrf += 1 / (60 + features["als_rank"])

            if features.get("from_sim"):
                rrf += 1 / (60 + features["sim_rank"])

            if features.get("from_pop"):
                rrf += 1 / (60 + features["pop_rank"])

            rows.append({
                "user_idx": int(user_idx),
                "movie_id": int(movie_id),

                "als_score": features.get("als_score", 0.0),
                "sim_score": features.get("sim_score", 0.0),
                "pop_score": features.get("pop_score", 0.0),

                "als_rank": features.get("als_rank", 0),
                "sim_rank": features.get("sim_rank", 0),
                "pop_rank": features.get("pop_rank", 0),

                "from_als": features.get("from_als", 0),
                "from_sim": features.get("from_sim", 0),
                "from_pop": features.get("from_pop", 0),

                "rrf": rrf
            })

    result = pd.DataFrame(rows)
    result["user_id"] = user_encoder.inverse_transform(result["user_idx"])
    return result


In [ ]:
def fit_retrieval(history, n_users):
    als_artifacts = fit_als(history, n_users=n_users, factors=16, regularization=0.5, iterations=10)
    sim_artifacts = fit_item_similarity(history, n_users=n_users)

    popularity_model = Recommender().fit(history)

    return als_artifacts, sim_artifacts, popularity_model

def make_candidates(history, target, candidate_k=500):
    als_artifacts, sim_artifacts, popularity_model = fit_retrieval(history, n_users=n_users)

    users = target["user_idx"].unique()
    candidates = build_candidates(
        users,
        history,
        als_artifacts,
        sim_artifacts,
        popularity_model,
        k_als=500,
        k_sim=1100,
        k_pop=300,
        max_sim_seeds=50
    )

    candidates = (
        candidates
        .sort_values(
            ["user_id", "rrf"],
            ascending=[True, False]
        )
        .groupby("user_id")
        .head(candidate_k)
        .reset_index(drop=True)
    )

    return candidates, (als_artifacts, sim_artifacts, popularity_model)


In [ ]:
def recall(candidates, target, min_rating=4):

    relevant = (
        target.loc[target["rating"] >= 4]
        .groupby("user_id")["movie_id"]
        .apply(set)
        .to_dict()
    )

    predicted = candidates.groupby("user_id")["movie_id"].apply(set).to_dict()

    recalls = []

    for user_id, y_true in relevant.items():
        if not y_true:
            continue

        y_pred = predicted.get(user_id, set())
        recalls.append(len(y_true & y_pred) / len(y_true))

    return np.mean(recalls) if recalls else 0.0

def source_candidate_recall(candidates, target):
    result = {
        "all": recall(candidates, target)
    }

    for source in ["als", "sim", "pop"]:
        source_candidates = candidates.loc[candidates[f"from_{source}"] == 1]
        result[source] = recall(source_candidates, target)

    return pd.Series(result, name="recall")


## Ranker dataset


In [ ]:
def build_ranker_dataset(
    candidates,
    history,
    target,
    df_users,
    df_movies,
    query_prefix
):
    target_labels = target[["user_id", "movie_id", "rating"]].copy()
    target_labels["label"] = np.where(
        target_labels["rating"] >= 4, 
        target_labels["rating"] - 2, 
        0
    )

    data = candidates.merge(
        target_labels[["user_id", "movie_id", "label"]],
        on=["user_id", "movie_id"],
        how="left"
    )

    data["label"] = data["label"].fillna(0).astype(np.int8)

    user_agg = (
        history.groupby("user_id")
        .agg(
            user_mean_rating=("rating", "mean"),
            user_rating_count=("rating", "count"),
            user_last_interaction=("timestamp", "max"),
            user_positive_rate=("rating", lambda x: (x >= 4).mean())
        )
        .reset_index()
    )

    movie_agg = (
        history.groupby("movie_id")
        .agg(
            movie_mean_rating=("rating", "mean"),
            movie_rating_count=("rating", "count"),
            movie_positive_rate=("rating", lambda x: (x >= 4).mean())
        )
        .reset_index()
    )

    data = data.merge(df_users, on="user_id", how="left")
    data = data.merge(
        df_movies.drop(columns=["title", "genres"], errors="ignore"),
        on="movie_id",
        how="left"
    )
    data = data.merge(user_agg, on="user_id", how="left")
    data = data.merge(movie_agg, on="movie_id", how="left")

    data["movie_mean_rating"] = data["movie_mean_rating"].fillna(history["rating"].mean())
    data["movie_rating_count"] = data["movie_rating_count"].fillna(0)
    data["movie_positive_rate"] = data["movie_positive_rate"].fillna(0)

    data["group_id"] = query_prefix + "_" + data["user_id"].astype(str)
    data = data.sort_values(["group_id", "movie_id"]).reset_index(drop=True)

    return data


In [ ]:
ID_COLUMNS = ["user_id", "user_idx", "movie_id", "group_id"]
DROP_COLUMNS = ID_COLUMNS + ["label", "zip_code", "title", "genre"]
CAT_FEATURES = ["gender", "age_group", "occupation"]


In [ ]:
def split_xy(data):
    X = data.drop(columns=DROP_COLUMNS, errors="ignore")
    y = data["label"]
    group_id = data["group_id"]

    return X, y, group_id

def keep_informative_groups(data):
    informative = (
        data.groupby("group_id")["label"]
        .max()
        .loc[lambda x: x > 0]
        .index
    )

    return data.loc[data["group_id"].isin(informative)].copy()


In [ ]:
def evaluate_ranker(model, data, target, k=10):

    X, _, _ = split_xy(data)
    scored = data[["user_id", "movie_id"]].copy()
    scored["score"] = model.predict(X)

    predictions = (
        scored.sort_values(["user_id", "score"], ascending=[True, False])
        .groupby("user_id")["movie_id"]
        .apply(lambda x: x.head(k).to_numpy())
        .to_dict()
    )

    relevant = (
        target.loc[target["rating"] >= 4]
        .groupby("user_id")["movie_id"]
        .apply(set)
        .to_dict()
    )

    recalls = []
    ndcgs = []

    for user_id, y_true in relevant.items():
        y_pred = predictions.get(user_id, [])

        recalls.append(recall_at_k(y_true, y_pred, k=k))
        ndcgs.append(ndcg_at_k(y_true, y_pred, k=k))

    return {
        f"recall@{k}": float(np.mean(recalls)),
        f"ndcg@{k}": float(np.mean(ndcgs)),
        "evaluated_users": len(ndcgs)
    }


## Train and validation candidates


In [ ]:
CANDIDATE_K = 500

train_candidates, train_retrieval = make_candidates(
    rank_history, rank_target, CANDIDATE_K
)
val_candidates, val_retrieval = make_candidates(
    val_history, val_target, CANDIDATE_K
)

print("Train retrieval recall")
print(source_candidate_recall(train_candidates, rank_target))
print("Validation retrieval recall")
print(source_candidate_recall(val_candidates, val_target))


In [ ]:
rank_train_df = build_ranker_dataset(
    train_candidates, rank_history, rank_target, df_users, df_movies, "rank_train"
)
rank_val_df = build_ranker_dataset(
    val_candidates, val_history, val_target, df_users, df_movies, "validation"
)

rank_train_pool_df = keep_informative_groups(rank_train_df)
rank_val_pool_df = keep_informative_groups(rank_val_df)

X_train, y_train, group_train = split_xy(rank_train_pool_df)
X_val, y_val, group_val = split_xy(rank_val_pool_df)

train_pool = Pool(X_train, y_train, group_id=group_train, cat_features=CAT_FEATURES)
val_pool = Pool(X_val, y_val, group_id=group_val, cat_features=CAT_FEATURES)

print("train groups:", rank_train_df["group_id"].nunique(), "->", rank_train_pool_df["group_id"].nunique())
print("val groups:", rank_val_df["group_id"].nunique(), "->", rank_val_pool_df["group_id"].nunique())


## Baseline ranker


In [ ]:
BASE_PARAMS = {
    "iterations": 1000,
    "learning_rate": 0.05,
    "depth": 7,
    "l2_leaf_reg": 4,
    "random_strength": 0.5,
}

model = CatBoostRanker(
    **BASE_PARAMS,
    loss_function="YetiRank",
    eval_metric="NDCG:top=10",
    random_seed=42,
    thread_count=6,
    verbose=100,
    task_type="GPU"
)
model.fit(
    train_pool,
    eval_set=val_pool,
    use_best_model=True,
    early_stopping_rounds=100
)

print("best iteration:", model.get_best_iteration())
print("best score:", model.get_best_score())
print(evaluate_ranker(model, rank_val_df, val_target, k=10))


## Hyperparameter search


In [ ]:
def objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 500, 2000),
        "learning_rate": 0.05,
        "depth": trial.suggest_int("depth", 4, 8),
        "l2_leaf_reg": trial.suggest_int("l2_leaf_reg", 1, 6),
        "random_strength": trial.suggest_float("random_strength", 0.1, 0.8),
    }

    trial_model = CatBoostRanker(
        **params,
        loss_function="YetiRank",
        eval_metric="NDCG:top=10",
        random_seed=42,
        thread_count=6,
        verbose=False,
        task_type="GPU"
    )
    trial_model.fit(
        train_pool,
        eval_set=val_pool,
        use_best_model=True,
        early_stopping_rounds=100
    )

    metrics = evaluate_ranker(trial_model, rank_val_df, val_target, k=10)
    return metrics["ndcg@10"]


In [ ]:
study = optuna.create_study(
    storage="sqlite:///catboost_ranker.db",
    study_name="CatBoostRanker_v1",
    direction="maximize",
    load_if_exists=True
)

study.optimize(objective, n_trials=100, show_progress_bar=True)
print(study.best_value)
print(study.best_params)


## Selected ranker


In [ ]:
selected_params = {
    **study.best_params,
    "learning_rate": 0.05,
}

selected_model = CatBoostRanker(
    **selected_params,
    loss_function="YetiRank",
    eval_metric="NDCG:top=10",
    random_seed=42,
    thread_count=6,
    verbose=100,
    task_type="GPU"
)
selected_model.fit(
    train_pool,
    eval_set=val_pool,
    use_best_model=True,
    early_stopping_rounds=100
)

best_iteration = selected_model.get_best_iteration() + 1
val_metrics = evaluate_ranker(selected_model, rank_val_df, val_target, k=10)
print("best iteration:", best_iteration)
print(val_metrics)


In [ ]:
feature_importance = (
    pd.DataFrame({
        "feature": X_train.columns,
        "importance": selected_model.get_feature_importance(train_pool)
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
feature_importance


## Final fit and test


In [ ]:
rank_fit_df = pd.concat(
    [rank_train_pool_df, rank_val_pool_df],
    ignore_index=True
).sort_values(["group_id", "movie_id"]).reset_index(drop=True)

X_fit, y_fit, group_fit = split_xy(rank_fit_df)
fit_pool = Pool(X_fit, y_fit, group_id=group_fit, cat_features=CAT_FEATURES)

final_params = {**selected_params, "iterations": best_iteration}
final_model = CatBoostRanker(
    **final_params,
    loss_function="YetiRank",
    eval_metric="NDCG:top=10",
    random_seed=42,
    thread_count=6,
    verbose=100,
    task_type="GPU"
)
final_model.fit(fit_pool)


In [ ]:
test_candidates, test_retrieval = make_candidates(
    test_history, test_target, CANDIDATE_K
)
test_rank_df = build_ranker_dataset(
    test_candidates, test_history, test_target, df_users, df_movies, "test"
)

print(source_candidate_recall(test_candidates, test_target))
test_metrics = evaluate_ranker(final_model, test_rank_df, test_target, k=10)
test_metrics


## Save artifacts


In [ ]:
ARTIFACT_DIR = ROOT_DIR / "notebook" / "data" / "models"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

final_model.save_model(ARTIFACT_DIR / "00_catboost_ranker.cbm")
joblib.dump(test_retrieval, ARTIFACT_DIR / "00_retrieval_test_history.pkl")
